# CXR Essential-Tag 평가 단일 파이프라인

기관의 **CXR DICOM 원본 폴더**로부터 **최종 품질평가 CSV 5개**까지 한 번에 산출하는 노트북입니다.
아래 **① 사용자 설정 셀**의 경로 변수만 본인 환경에 맞게 수정한 뒤,
셀을 **위에서부터 순서대로** 실행하세요.

## 파이프라인 개요
```
DICOM(.dcm) 폴더
   │  ② dicom_to_parquet.py  (메타데이터 태그 추출)
   ▼
hive 파티션 parquet dataset  (modality=.../tag=.../part-*.parquet)
   │  ③ 단일 parquet 병합
   ▼
단일 .parquet (long-format)
   │  ④ df_input 정제  (컬럼/스키마 변환 · 최상위 태그 추출 · 다중값 병합)
   ▼
Evaluator df_input
   │  ⑤ CxrEssentialTagEvaluator 평가
   ▼
최종 CSV 5개
```

## 산출되는 최종 CSV 5개 (`OUTPUT_DIR`)
1. `cxr_overall_rates.csv` — 태그별 completeness / conformance
2. `cxr_conformance_summary.csv` — CS 값 pass/partial/none 요약
3. `cxr_conformance_partial.csv` — 부분 일치(허용값을 단어로 포함) 상세
4. `cxr_conformance_none.csv` — 전혀 불일치 상세
5. `cxr_unconformed_values.csv` — 미준수 unique value 통합

> 참고 코드
> - 메타데이터 추출: <https://github.com/dr-you-group/dicom_to_parquet>
> - 품질 평가: <https://github.com/dr-you-group/dicomHeterogeneity>


## ① 사용자 설정 단계
**이 셀의 경로 변수만 로컬 환경에 맞게 수정**하면 됩니다. (나머지 셀은 수정 불필요)

- Windows/Unix 모두 호환되도록 `pathlib.Path` 를 사용합니다.
- `MODALITY_FILTER` : CXR 만 남기기 위한 modality 필터. 빈 리스트(`[]`)면 전체 modality 사용.


In [ ]:
from pathlib import Path

# ===== 기관에서 수정해야 하는 경로 (여기만 입력해주시면 됩니다!) =====
DICOM_ROOT          = Path() # 기관 CXR DICOM(.dcm) 최상위 폴더 (하위 재귀 탐색)
WORK_DIR            = Path() # 처음 공유드린 두 git repo clone 및 중간 산출물 폴더

PARQUET_OUT_DIR     = Path() # dicom_to_parquet 파티션 dataset 저장 경로 (폴더 단위로 나눠서 저장됨)
SINGLE_PARQUET_PATH = Path() # 병합한 단일 .parquet 경로
OUTPUT_DIR          = Path() # (수합 요청드린) 최종 CSV 5개 저장 폴더

# CXR 만 필터 (빈 리스트면 전체 modality 사용). CR=Computed Radiography, DX=Digital Radiography
MODALITY_FILTER     = ["CR", "DX"]

# 필요한 폴더 생성 (없으면 만들고, 있으면 그대로 두는 코드)
for _p in (WORK_DIR, PARQUET_OUT_DIR, SINGLE_PARQUET_PATH.parent, OUTPUT_DIR):
    _p.mkdir(parents=True, exist_ok=True)

print("설정 완료")
print(f"  DICOM_ROOT          = {DICOM_ROOT}")
print(f"  WORK_DIR            = {WORK_DIR}")
print(f"  PARQUET_OUT_DIR     = {PARQUET_OUT_DIR}")
print(f"  SINGLE_PARQUET_PATH = {SINGLE_PARQUET_PATH}")
print(f"  OUTPUT_DIR          = {OUTPUT_DIR}")
print(f"  MODALITY_FILTER     = {MODALITY_FILTER}")

# DICOM_ROOT 존재/내용 확인
if not DICOM_ROOT.exists():
    raise FileNotFoundError(f"[오류] DICOM_ROOT 폴더가 존재하지 않습니다: {DICOM_ROOT}")
_n_dcm = sum(1 for _ in DICOM_ROOT.rglob("*.dcm"))
print(f"\nDICOM_ROOT 하위 .dcm 파일 수: {_n_dcm}개")
if _n_dcm == 0:
    raise FileNotFoundError(
        f"[오류] DICOM_ROOT 하위에 .dcm 파일이 없습니다: {DICOM_ROOT}\n"
        f"       경로가 맞는지, 확장자가 .dcm 인지 확인하세요.")


설정 완료
  DICOM_ROOT          = C:\Projects\dicomHeterogeneity_with_git\cxr-sample-dir
  WORK_DIR            = C:\Projects\dicomHeterogeneity_with_git
  PARQUET_OUT_DIR     = C:\Projects\dicomHeterogeneity_with_git\_parquet_out
  SINGLE_PARQUET_PATH = C:\Projects\dicomHeterogeneity_with_git\_single\cxr_all.parquet
  OUTPUT_DIR          = C:\Projects\dicomHeterogeneity_with_git\_output
  MODALITY_FILTER     = ['CR', 'DX']

DICOM_ROOT 하위 .dcm 파일 수: 1개


## ② 환경 준비 단계
필요한 파이썬 패키지를 확인하고, 두 참조 레포(`dicom_to_parquet`, `dicomHeterogeneity`)를
`WORK_DIR` 에 준비합니다.

- 패키지 설치가 필요하신 경우, 아래 `%pip install` 주석을 해제해 실행해주시면 감사하겠습니다.
- `WORK_DIR` 에 공유드린 두 개의 git repo를 이미 clone하신 경우는 clone을 skip하고, 폴더가 없다면 여기서 git clone 진행합니다.

In [2]:
# 필요 패키지 (설치가 필요하면 아래 주석 해제)
# %pip install pydicom pyarrow pandas numpy openpyxl

import sys, subprocess, json

# --- 필수 패키지 import 확인 ---
try:
    import pydicom, pyarrow, pandas as pd, numpy as np, openpyxl  # noqa: F401
    import pyarrow.dataset as ds
    import pyarrow.parquet as pq
    print("패키지 import 성공")
    print(f"  python  = {sys.version.split()[0]}")
    print(f"  pydicom = {pydicom.__version__}, pyarrow = {pyarrow.__version__}, pandas = {pd.__version__}")
except ImportError as e:
    raise ImportError(
        f"[오류] 필수 패키지가 없습니다: {e}\n"
        f"       위 셀의 '%pip install pydicom pyarrow pandas numpy openpyxl' 주석을 해제해 설치하세요.")

# --- 두 레포 준비 (없으면 git clone, 있으면 skip) ---
REPOS = {
    "dicom_to_parquet":   "https://github.com/dr-you-group/dicom_to_parquet.git",
    "dicomHeterogeneity": "https://github.com/dr-you-group/dicomHeterogeneity.git",
}
for name, url in REPOS.items():
    target = WORK_DIR / name
    if target.exists():
        print(f"[skip] 이미 존재: {target}")
        continue
    print(f"[clone] {url} -> {target}")
    r = subprocess.run(["git", "clone", "--depth", "1", url, str(target)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(
            f"[오류] git clone 실패: {url}\n{r.stderr}\n"
            f"       인터넷/깃 설치 상태를 확인하거나, 레포를 수동으로 {target} 에 복사하세요.")
    print("       clone 완료")

# 핵심 파일 존재 확인
DICOM_TO_PARQUET_PY = WORK_DIR / "dicom_to_parquet" / "dicom_to_parquet.py"
EVALUATOR_DIR       = WORK_DIR / "dicomHeterogeneity" / "DicomStandardEvaluator" / "Evaluator"
REFERENCE_XLSX      = WORK_DIR / "dicomHeterogeneity" / "files" / "CxrEssentialTags" / "CxrEssentialTags_ReferenceSet.xlsx"
for p in (DICOM_TO_PARQUET_PY, EVALUATOR_DIR, REFERENCE_XLSX):
    if not p.exists():
        raise FileNotFoundError(f"[오류] 필요한 파일/폴더가 없습니다: {p}")
print("\n참조 레포/파일 확인 완료")


패키지 import 성공
  python  = 3.12.4
  pydicom = 3.0.0, pyarrow = 14.0.2, pandas = 2.2.2
[skip] 이미 존재: C:\Projects\dicomHeterogeneity_with_git\dicom_to_parquet
[skip] 이미 존재: C:\Projects\dicomHeterogeneity_with_git\dicomHeterogeneity

참조 레포/파일 확인 완료


## ③ 메타데이터 추출 단계
`dicom_to_parquet.py` 를 `subprocess` 로 실행해 DICOM 헤더의 모든 태그를
long-format 으로 펼친 **hive 파티션 parquet dataset** 을 `PARQUET_OUT_DIR` 에 생성합니다.

- 옵션: `--dicom_root DICOM_ROOT --out_dir PARQUET_OUT_DIR --skip_pixel_data`
  (`--skip_pixel_data` 로 픽셀 데이터를 제외해 빠르게 처리)
- 실행 후 `_build_summary.json` 을 출력해 **처리 파일 수 / 실패 수**를 확인합니다.


In [3]:
# dicom_to_parquet.py 실행 (현재 커널과 동일한 python 사용: sys.executable)
cmd = [
    sys.executable, str(DICOM_TO_PARQUET_PY),
    "--dicom_root", str(DICOM_ROOT),
    "--out_dir",    str(PARQUET_OUT_DIR),
    "--skip_pixel_data",
]
print("실행 명령:\n  " + " ".join(cmd) + "\n")
proc = subprocess.run(cmd, capture_output=True, text=True)
print("----- STDOUT -----")
print(proc.stdout[-4000:])
if proc.stderr.strip():
    print("----- STDERR (마지막 2000자) -----")
    print(proc.stderr[-2000:])
if proc.returncode != 0:
    raise RuntimeError(f"[오류] dicom_to_parquet 실행 실패 (returncode={proc.returncode}). 위 STDERR 확인.")

# _build_summary.json 확인
summary_path = PARQUET_OUT_DIR / "_build_summary.json"
if not summary_path.exists():
    raise FileNotFoundError(f"[오류] 요약 파일이 없습니다: {summary_path} (추출이 정상 완료되지 않음)")
build_summary = json.loads(summary_path.read_text(encoding="utf-8"))
print("\n===== _build_summary.json =====")
print(json.dumps(build_summary, ensure_ascii=False, indent=2))
if build_summary.get("files_processed", 0) == 0:
    raise RuntimeError("[오류] 처리된 파일이 0개입니다. DICOM_ROOT 경로/내용을 확인하세요.")
if build_summary.get("rows_total", 0) == 0:
    raise RuntimeError("[오류] 추출된 태그 행이 0개입니다. DICOM 파일이 유효한지 확인하세요.")


실행 명령:
  C:\Users\Kyulee Jeon\anaconda3\python.exe C:\Projects\dicomHeterogeneity_with_git\dicom_to_parquet\dicom_to_parquet.py --dicom_root C:\Projects\dicomHeterogeneity_with_git\cxr-sample-dir --out_dir C:\Projects\dicomHeterogeneity_with_git\_parquet_out --skip_pixel_data

----- STDOUT -----
[DONE]
{
  "dicom_root": "C:\\Projects\\dicomHeterogeneity_with_git\\cxr-sample-dir",
  "out_dir": "C:\\Projects\\dicomHeterogeneity_with_git\\_parquet_out",
  "files_processed": 1,
  "files_failed": 0,
  "rows_total": 151,
  "batches_written": 1,
  "skip_pixel_data": true,
  "skip_tags": [
    "54001010",
    "7FE00010"
  ],
  "compression": "zstd",
  "schema": [
    "file_path:string",
    "study_uid:string",
    "series_uid:string",
    "instance_uid:string",
    "sop_class_uid:string",
    "modality:string",
    "tag:string",
    "vr:string",
    "vm:string",
    "path:string",
    "value:string"
  ]
}


===== _build_summary.json =====
{
  "dicom_root": "C:\\Projects\\dicomHeterogeneity_wit

## ④ 단일 parquet 병합 단계
파티션 dataset 전체를 읽어 하나의 `.parquet` 파일(`SINGLE_PARQUET_PATH`)로 병합하고,
DataFrame 으로도 로드합니다.

> **참고**: `dicom_to_parquet` 출력은 (서버 용량 관리를 위해) `modality=.../tag=.../part-*.parquet` 형태의 **hive 파티션**입니다.
> 따라서 `pyarrow.dataset(..., partitioning="hive")` 로 읽어야 파티션 컬럼(`modality`, `tag`)이 복원됩니다.
> (일반 `read_parquet` 으로 개별 파일을 읽으면 `modality`/`tag` 컬럼이 사라집니다.)
> 또한 `tag` 값이 `00080016` 처럼 숫자로만 이뤄지면 정수로 잘못 추론되어 앞자리 0이 사라질 수 있으므로,
> 파티션 스키마를 **문자열(string)** 로 명시해 읽습니다.


In [4]:
import pyarrow as pa

# 파티션 컬럼을 문자열로 명시 (leading-zero 손실 방지)
hive_part = ds.partitioning(
    pa.schema([("modality", pa.string()), ("tag", pa.string())]),
    flavor="hive",
)
dataset = ds.dataset(str(PARQUET_OUT_DIR), format="parquet",
                     partitioning=hive_part, exclude_invalid_files=True)

# 병합 결과에서 기대하는 컬럼(11개) — 파일 컬럼 9개 + 파티션 컬럼 2개(modality, tag)
EXPECTED_COLS = ["file_path", "study_uid", "series_uid", "instance_uid",
                 "sop_class_uid", "modality", "tag", "vr", "vm", "path", "value"]

# 대용량 대비: batch 단위로 읽어 단일 parquet 파일로 스트리밍 저장
SINGLE_PARQUET_PATH.parent.mkdir(parents=True, exist_ok=True)
writer = None
n_rows = 0
try:
    for batch in dataset.to_batches(batch_size=200_000):
        if batch.num_rows == 0:
            continue
        tbl = pa.Table.from_batches([batch])
        # 컬럼 순서 정렬 (존재하는 것만)
        cols = [c for c in EXPECTED_COLS if c in tbl.column_names]
        tbl = tbl.select(cols)
        if writer is None:
            writer = pq.ParquetWriter(str(SINGLE_PARQUET_PATH), tbl.schema)
        writer.write_table(tbl)
        n_rows += tbl.num_rows
finally:
    if writer is not None:
        writer.close()

if n_rows == 0:
    raise RuntimeError(f"[오류] 병합할 데이터가 없습니다. {PARQUET_OUT_DIR} 안의 parquet 파일을 확인하세요.")
print(f"단일 parquet 저장 완료: {SINGLE_PARQUET_PATH}  (총 {n_rows:,} 행)")

# DataFrame 으로 로드해 확인
df_raw = pd.read_parquet(SINGLE_PARQUET_PATH)
# tag/modality 는 항상 문자열로 통일 (tag 는 8자리 대문자 hex 로 정규화)
df_raw["tag"] = df_raw["tag"].astype(str).str.upper().str.zfill(8)
df_raw["modality"] = df_raw["modality"].astype(str)
print(f"\ndf_raw shape = {df_raw.shape}")
print("컬럼:", df_raw.columns.tolist())
print("\nmodality 별 행 수:")
print(df_raw["modality"].value_counts().to_string())
print("\nhead():")
print(df_raw.head().to_string())


단일 parquet 저장 완료: C:\Projects\dicomHeterogeneity_with_git\_single\cxr_all.parquet  (총 151 행)

df_raw shape = (151, 11)
컬럼: ['file_path', 'study_uid', 'series_uid', 'instance_uid', 'sop_class_uid', 'modality', 'tag', 'vr', 'vm', 'path', 'value']

modality 별 행 수:
modality
DX    151

head():
                                                                                                 file_path                                     study_uid                                    series_uid                                  instance_uid                sop_class_uid modality       tag  vr vm          path                                         value
0  C:\Projects\dicomHeterogeneity_with_git\cxr-sample-dir\c48f357f-104d4294-115bacfa-b6ca836e-93b9a7f4.dcm  2.25.339234018222862501477993377722989025995  2.25.282741301623444726298556964625855490362  2.25.261272270659344861710025109691584643950  1.2.840.10008.5.1.4.1.1.1.1       DX  00080005  CS  1  00080005[0]/                                    I

## ⑤ df_input 정제 단계
`dicom_to_parquet` 출력 스키마를 Evaluator 가 요구하는 `df_input` 스키마로 변환합니다.

### 스키마 매핑
| df_input 컬럼 | 생성 규칙 |
|---|---|
| `IOD` | `sop_class_uid` → `pydicom.uid.UID(uid).name` (실패 시 UID 원문 유지) |
| `study_instance_uid` | `study_uid` 그대로 |
| `series_instance_uid` | `series_uid` 그대로 |
| `Manufacturer` | 같은 instance 의 tag `00080070` 값 (없으면 같은 series 값, 그래도 없으면 "") |
| `ScannerModel` | 같은 방식으로 tag `00081090` 값 |
| `Tag` | `tag` (8자리 대문자 hex) |
| `AttributeName` | `pydicom.datadict.keyword_for_tag(int(tag,16))` (없으면 "") |
| `Value` | 아래 전처리를 거친 값 |

### Value 전처리 규칙
1. `MODALITY_FILTER` 적용 (modality 기준).
2. **최상위 태그만 사용**: `path` 가 `"XXXXXXXX[i]/"` 형태(중첩 없음)인 행만 남김.
   → SQ 내부 중첩 행(`path` 에 `[` 2회 이상)과 SQ 컨테이너 행(value 가 `"SQ[n]"`)은 제외.
3. **다중값(VM>1) 병합**: 같은 `(instance_uid, tag)` 에 값이 여러 행이면
   `"['A', 'B']"` 형태의 리스트 문자열 한 행으로 합침. (Evaluator 가 리스트 문자열을 파싱)


In [5]:
import re
from pydicom.uid import UID
from pydicom.datadict import keyword_for_tag

MANUFACTURER_TAG = "00080070"   # (0008,0070) Manufacturer
SCANNERMODEL_TAG = "00081090"   # (0008,1090) Manufacturer's Model Name

df = df_raw.copy()

# --- 규칙 1: MODALITY_FILTER 적용 ---
if MODALITY_FILTER:
    before = len(df)
    df = df[df["modality"].isin(MODALITY_FILTER)].copy()
    print(f"[1] MODALITY_FILTER={MODALITY_FILTER} 적용: {before:,} -> {len(df):,} 행")
    if df.empty:
        raise RuntimeError(
            f"[오류] MODALITY_FILTER={MODALITY_FILTER} 결과 데이터가 0행입니다.\n"
            f"       실제 modality 값을 확인하세요: {sorted(df_raw['modality'].unique())}")
else:
    print("[1] MODALITY_FILTER 비어있음 -> 전체 modality 사용")

# --- 규칙 2: 최상위 태그(비중첩)만 사용 ---
# top-level scalar 의 path 형태: 'XXXXXXXX[i]/'  (대괄호 인덱스 1개, 중첩 없음)
#   - SQ 컨테이너 행: path 'XXXXXXXX/' (대괄호 없음) & value 'SQ[n]'  -> 제외
#   - SQ 내부 중첩 행: path 'AAAAAAAA[i]/BBBBBBBB[j]/' (대괄호 2쌍 이상) -> 제외
TOP_LEVEL_RE = re.compile(r"^[0-9A-Fa-f]{8}\[\d+\]/$")
top_mask = df["path"].astype(str).str.match(TOP_LEVEL_RE)
# 방어적으로 SQ 컨테이너 값('SQ[n]')도 제외
sq_mask = df["value"].astype(str).str.match(r"^SQ\[\d+\]$")
top_mask = top_mask & (~sq_mask)
before = len(df)
df = df[top_mask].copy()
print(f"[2] 최상위 태그만 사용: {before:,} -> {len(df):,} 행")
if df.empty:
    raise RuntimeError("[오류] 최상위 태그 필터 결과 0행입니다. path 형식을 확인하세요.")

# --- Manufacturer / ScannerModel 조회 테이블 (instance -> series 폴백) ---
def _nonempty(s):
    s = "" if s is None else str(s).strip()
    return s if s not in ("", "nan", "[]") else ""

def build_lookup(tag_code):
    sub = df[df["tag"] == tag_code][["instance_uid", "series_uid", "value"]].copy()
    sub["value"] = sub["value"].map(_nonempty)
    sub = sub[sub["value"] != ""]
    inst_map = sub.groupby("instance_uid")["value"].first().to_dict()   # instance 단위
    ser_map  = sub.groupby("series_uid")["value"].first().to_dict()     # series 단위(폴백)
    return inst_map, ser_map

manu_inst, manu_ser   = build_lookup(MANUFACTURER_TAG)
model_inst, model_ser = build_lookup(SCANNERMODEL_TAG)

def resolve(row, inst_map, ser_map):
    v = inst_map.get(row["instance_uid"])
    if v:
        return v
    v = ser_map.get(row["series_uid"])
    return v if v else ""

# --- 규칙 3: 다중값(VM>1) 병합: (instance_uid, tag) 단위로 value 를 하나로 합침 ---
df["_idx"] = df["path"].astype(str).str.extract(r"\[(\d+)\]").astype(int)  # 다중값 순서
grp_keys = ["instance_uid", "study_uid", "series_uid", "sop_class_uid", "tag"]

def merge_values(g):
    vals = g.sort_values("_idx")["value"].astype(str).tolist()
    if len(vals) == 1:
        return vals[0]                 # 단일값: 그대로
    return str(vals)                   # 다중값: "['A', 'B']" 리스트 문자열

merged = (df.groupby(grp_keys, sort=False)
            .apply(merge_values, include_groups=False)
            .rename("Value").reset_index())
print(f"[3] 다중값 병합 후 (instance x tag) 행: {len(merged):,}")

# --- 스키마 매핑하여 df_input 생성 ---
def uid_to_iod(uid):
    try:
        name = UID(str(uid)).name
        return name if name else str(uid)
    except Exception:
        return str(uid)

def tag_to_keyword(tag8):
    try:
        kw = keyword_for_tag(int(tag8, 16))
        return kw if kw else ""
    except Exception:
        return ""

df_input = pd.DataFrame({
    "IOD":                 merged["sop_class_uid"].map(uid_to_iod),
    "study_instance_uid":  merged["study_uid"],
    "series_instance_uid": merged["series_uid"],
    "Manufacturer":        merged.apply(lambda r: resolve(r, manu_inst, manu_ser), axis=1),
    "ScannerModel":        merged.apply(lambda r: resolve(r, model_inst, model_ser), axis=1),
    "Tag":                 merged["tag"],
    "AttributeName":       merged["tag"].map(tag_to_keyword),
    "Value":               merged["Value"],
})

# 저장 (utf-8-sig: Excel 한글 호환)
df_input_path = OUTPUT_DIR / "df_input.csv"
df_input.to_csv(df_input_path, index=False, encoding="utf-8-sig")
print(f"\ndf_input 저장: {df_input_path}")
print(f"df_input shape = {df_input.shape}")
print("컬럼:", df_input.columns.tolist())
print("\nhead():")
print(df_input.head(15).to_string())


[1] MODALITY_FILTER=['CR', 'DX'] 적용: 151 -> 151 행
[2] 최상위 태그만 사용: 151 -> 100 행
[3] 다중값 병합 후 (instance x tag) 행: 99

df_input 저장: C:\Projects\dicomHeterogeneity_with_git\_output\df_input.csv
df_input shape = (99, 8)
컬럼: ['IOD', 'study_instance_uid', 'series_instance_uid', 'Manufacturer', 'ScannerModel', 'Tag', 'AttributeName', 'Value']

head():
                                               IOD                            study_instance_uid                           series_instance_uid Manufacturer ScannerModel       Tag           AttributeName                                         Value
0   Digital X-Ray Image Storage - For Presentation  2.25.339234018222862501477993377722989025995  2.25.282741301623444726298556964625855490362                            00080005    SpecificCharacterSet                                    ISO_IR 100
1   Digital X-Ray Image Storage - For Presentation  2.25.339234018222862501477993377722989025995  2.25.282741301623444726298556964625855490362              

## ⑥ 품질 평가 실행 단계
**무엇을**: 정제한 `df_input` 과 표준 참조 세트를 평가기에 넣어 **series 단위** 품질 지표를 계산하고,
최종 CSV 5개를 `OUTPUT_DIR` 에 저장합니다.
**왜**: essential-tag 기준의 tag_completeness / value_completeness / value_conformance 를 산출하기 위해서입니다.

- 모든 CSV 는 `encoding="utf-8-sig"`, `group_cols=None`(전체 데이터셋 1개 그룹) 으로 저장합니다.


In [6]:
sys.path.append(str(WORK_DIR / "dicomHeterogeneity" / "DicomStandardEvaluator" / "Evaluator"))
from CxrEssentialTagEvaluator import CxrEssentialTagEvaluator

# 표준 참조 세트 로드
df_standard = pd.read_excel(REFERENCE_XLSX)
print(f"표준 참조 세트 로드: {REFERENCE_XLSX}")
print(f"  standard shape = {df_standard.shape}, 태그 수 = {df_standard['Tag'].nunique()}")

evaluator = CxrEssentialTagEvaluator(df_input, df_standard)

# 1) 전체 completeness/conformance
overall = evaluator.analyze(group_cols=None)
overall_path = OUTPUT_DIR / "cxr_overall_rates.csv"
overall.to_csv(overall_path, index=False, encoding="utf-8-sig")
print(f"\n[1/5] 저장: {overall_path}  (shape={overall.shape})")

# 2~4) conformance 세부 리포트 3종 (summary / partial / none)
conf_prefix = OUTPUT_DIR / "cxr_conformance"
report = evaluator.export_conformance_subreport(str(conf_prefix), group_cols=None)
print(f"[2/5] 저장: {conf_prefix}_summary.csv  (shape={report['summary'].shape})")
print(f"[3/5] 저장: {conf_prefix}_partial.csv  (shape={report['partial'].shape})")
print(f"[4/5] 저장: {conf_prefix}_none.csv     (shape={report['none'].shape})")

# 5) 미준수 unique value 통합
unconf_path = OUTPUT_DIR / "cxr_unconformed_values.csv"
df_unconf = evaluator.export_unconformed_values(str(unconf_path), group_cols=None)
print(f"[5/5] 저장: {unconf_path}  (shape={df_unconf.shape})")


표준 참조 세트 로드: C:\Projects\dicomHeterogeneity_with_git\dicomHeterogeneity\files\CxrEssentialTags\CxrEssentialTags_ReferenceSet.xlsx
  standard shape = (28, 12), 태그 수 = 28

[1/5] 저장: C:\Projects\dicomHeterogeneity_with_git\_output\cxr_overall_rates.csv  (shape=(28, 12))
[2/5] 저장: C:\Projects\dicomHeterogeneity_with_git\_output\cxr_conformance_summary.csv  (shape=(8, 9))
[3/5] 저장: C:\Projects\dicomHeterogeneity_with_git\_output\cxr_conformance_partial.csv  (shape=(0, 7))
[4/5] 저장: C:\Projects\dicomHeterogeneity_with_git\_output\cxr_conformance_none.csv     (shape=(1, 7))
[5/5] 저장: C:\Projects\dicomHeterogeneity_with_git\_output\cxr_unconformed_values.csv  (shape=(1, 7))


## ⑦ 검증 단계
**무엇을**: 최종 CSV 5개의 존재 여부·행 수·컬럼명을 표로 확인하고,
`cxr_overall_rates` 의 핵심 지표(tag_completeness / value_completeness / value_conformance)를 요약합니다.
**왜**: 파이프라인이 끝까지 정상 산출됐는지 한눈에 점검하기 위해서입니다.


In [7]:
EXPECTED_OUTPUTS = [
    "cxr_overall_rates.csv",
    "cxr_conformance_summary.csv",
    "cxr_conformance_partial.csv",
    "cxr_conformance_none.csv",
    "cxr_unconformed_values.csv",
]

print("===== 최종 CSV 5개 검증 =====")
check_rows = []
all_ok = True
for fn in EXPECTED_OUTPUTS:
    p = OUTPUT_DIR / fn
    if p.exists():
        _df = pd.read_csv(p)
        check_rows.append({"file": fn, "exists": "O", "n_rows": len(_df),
                           "n_cols": _df.shape[1], "columns": ", ".join(map(str, _df.columns))[:80]})
    else:
        all_ok = False
        check_rows.append({"file": fn, "exists": "X", "n_rows": "-",
                           "n_cols": "-", "columns": "(파일 없음)"})
check_df = pd.DataFrame(check_rows)
print(check_df.to_string(index=False))

if not all_ok:
    raise RuntimeError("[오류] 일부 최종 CSV 가 생성되지 않았습니다. 위 표에서 'X' 항목을 확인하세요.")
print("\n[OK] 5개 CSV 모두 정상 생성되었습니다.")

# --- cxr_overall_rates 핵심 지표 요약 ---
print("\n===== cxr_overall_rates 핵심 지표 요약 =====")
ov = pd.read_csv(OUTPUT_DIR / "cxr_overall_rates.csv")
metric_cols = ["tag_completeness", "value_completeness", "value_conformance"]
total_series = int(ov["total_series"].iloc[0]) if len(ov) else 0
print(f"전체 series 수(total_series): {total_series}")
print(f"평가 태그 수: {len(ov)}")
print("\n[지표 평균 (태그 전체 기준)]")
print(ov[metric_cols].mean(numeric_only=True).round(4).to_string())

print("\n[태그별 지표 (상위 20개)]")
show_cols = ["Tag", "Attribute Name", "VR", "total_series", "series_with_tag",
             "tag_completeness", "value_completeness", "value_conformance"]
show_cols = [c for c in show_cols if c in ov.columns]
with pd.option_context("display.max_rows", 40, "display.width", 200):
    print(ov[show_cols].head(20).to_string(index=False))


===== 최종 CSV 5개 검증 =====
                       file exists  n_rows  n_cols                                                                          columns
      cxr_overall_rates.csv      O      28      12 Tag, Attribute Name, VR, total_series, series_with_tag, series_with_value, serie
cxr_conformance_summary.csv      O       8       9 Tag, Attribute Name, n_values, n_pass, n_partial, n_none, pct_pass, pct_partial,
cxr_conformance_partial.csv      O       0       7                 Tag, Attribute Name, Defined Values, Value, count, n_series, pct
   cxr_conformance_none.csv      O       1       7                 Tag, Attribute Name, Defined Values, Value, count, n_series, pct
 cxr_unconformed_values.csv      O       1       7 Tag, Attribute Name, Defined Values, Unconformed Value, category, count, n_serie

[OK] 5개 CSV 모두 정상 생성되었습니다.

===== cxr_overall_rates 핵심 지표 요약 =====
전체 series 수(total_series): 1
평가 태그 수: 28

[지표 평균 (태그 전체 기준)]
tag_completeness      0.8929
value_completeness    0.9